In [2]:
import numpy as np
import pandas as pd
import scanpy as sc

asthma_path = "./asthma/asthma_data_filtered.h5ad"


/data2/project/bin_jip/miniconda3/envs/biomarker/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
asthma=sc.read_h5ad(asthma_path)
"obs column별로 value counts, value의 개수가 10개 넘으면 출력하지 말기"
for col in asthma.obs.columns:
    value_counts = asthma.obs[col].value_counts()
    if len(value_counts) <= 30:
        print(f"Column: {col}")
        print(value_counts)

Column: batch
batch
batch2    454379
batch1     73355
Name: count, dtype: int64
Column: PRISM_ID
PRISM_ID
01-016    37883
01-026    35664
01-039    34139
01-014    33297
01-037    33261
01-011    33042
01-030    32672
01-031    32657
01-034    31816
01-015    31627
01-040    29836
01-032    28073
01-006    23498
01-007    16724
01-025    12198
01-008     9839
01-019     9572
01-004     7992
01-013     7403
01-012     7313
01-022     7163
01-023     6930
01-001     5906
01-002     5378
01-005     4341
01-020     3871
01-028     2894
01-018     2745
Name: count, dtype: int64
Column: Biologics
Biologics
reslizumab     318616
dupilumab      186138
mepolizumab     22980
Name: count, dtype: int64
Column: TimePoint
TimePoint
6M          268182
Baseline    259552
Name: count, dtype: int64
Column: RNA_snn_res.0.3
RNA_snn_res.0.3
0     107873
1      78668
2      69884
3      67046
4      43666
5      41885
6      29037
7      28249
8      21528
9      11850
10      9224
11      7501
12      5297

In [4]:
"""
'TimePoint' column의 value가 'Baseline'인 행들만 필터링하여 새로운 AnnData 객체로 저장
그리고 환자의 수 유지되는지 확인 'PRISM_ID' column이 환자 id임
Response는 1이면 1, 2,3,4면 0으로 매핑해서 'Response2' column 대체
"""
asthma_baseline = asthma[asthma.obs['TimePoint'] == 'Baseline'].copy()
print(f"Original number of patients: {asthma.obs['PRISM_ID'].nunique()}")
print(f"Baseline number of patients: {asthma_baseline.obs['PRISM_ID'].nunique()}")  
asthma_baseline.obs['Response2'] = asthma_baseline.obs['Response'].map({1: 1, 2: 0, 3: 0, 4: 0})
print(asthma_baseline.obs['Response2'].value_counts())


Original number of patients: 28
Baseline number of patients: 28
Response2
1    167534
0     92018
Name: count, dtype: int64


In [11]:
for col in asthma_baseline.obs.columns:
    value_counts = asthma_baseline.obs[col].value_counts()
    if len(value_counts) <= 30:
        print(f"Column: {col}")
        print(value_counts)
"""이제 asthma_baseline을 './asthma_baseline.filtered.h5ad'로 저장 """
asthma_baseline.write_h5ad('./asthma/asthma_baseline_filtered.h5ad')

Column: Sample_ID
Sample_ID
RH011013    19254
RH011022    18353
RH011036    17775
RH011034    17516
RH011031    16689
RH011009    16519
RH011011    16190
RH011012    16119
RH011028    15934
RH011026    15548
RH011035    14569
RH011007    11672
RH011029    10793
RH011006     8484
RH011018     4627
RH011017     4243
RH011004     4215
RH011019     3871
RH011010     3817
RH011014     3588
RH011005     3585
RH011020     2956
RH011024     2894
RH011016     2745
RH011021     2642
RH011001     1750
RH011002     1740
RH011023     1464
Name: count, dtype: int64
Column: batch
batch
batch2    221094
batch1     38458
Name: count, dtype: int64
Column: PRISM_ID
PRISM_ID
01-016    19254
01-026    18353
01-037    17775
01-039    17516
01-034    16689
01-011    16519
01-014    16190
01-015    16119
01-031    15934
01-030    15548
01-040    14569
01-007    11672
01-032    10793
01-006     8484
01-019     4627
01-012     4243
01-004     4215
01-020     3871
01-013     3817
01-008     3588
01-005     3585


In [12]:
"""
"Response2" column의 value인 1과 0 중 "PRISM_ID"에 mapping된 개수를 출력
"""

response_counts = asthma_baseline.obs.groupby('PRISM_ID')['Response2'].first().value_counts()


/tmp/ipykernel_922689/516252360.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  response_counts = asthma_baseline.obs.groupby('PRISM_ID')['Response2'].first().value_counts()


In [13]:
print(response_counts)

Response2
1    17
0    11
Name: count, dtype: int64


In [7]:
""" patient당 celltype 분포를 확인하자."""
patient_celltype_counts = asthma_baseline.obs.groupby(['PRISM_ID', 'CellType_minor']).size().unstack(fill_value=0)
print(patient_celltype_counts)

CellType_minor  B Memory  B Naive  CD4 Naive  CD4 TCM  CD4 TEM  CD8 Naive  \
PRISM_ID                                                                    
01-001                79       72        335        6      152        153   
01-002                24       54        338       61      473        120   
01-004               105       93        432      198      178        144   
01-005               289      449        710       40      145        466   
01-006               308      848       1396      568      680        387   
01-007               514      180       2874     1083      757        608   
01-008               227      122        610      230      229         85   
01-011               731      526       2521      700     1175        102   
01-012                83       77        401      181      443         35   
01-013               226      254        408      203      333         39   
01-014               814     1275       1817     1681     1741        352   

/tmp/ipykernel_1174525/3502136864.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  patient_celltype_counts = asthma_baseline.obs.groupby(['PRISM_ID', 'CellType_minor']).size().unstack(fill_value=0)


In [1]:
""""celltype 이 doublet인 cell이 몇개인지 확인하고 삭제하자 그리고 저장하자"""

import scanpy as sc
asthma_baseline = sc.read_h5ad('./asthma/asthma_baseline_filtered.h5ad')

doublet_count = asthma_baseline.obs['CellType_minor'].value_counts().get('Doublet', 0)
print(f"Number of cells labeled as 'Doublet': {doublet_count}")
asthma_baseline = asthma_baseline[asthma_baseline.obs['CellType_minor'] != 'Doublet'].copy()
print(f"Number of cells after removing 'Doublet': {asthma_baseline.n_obs}")

/data2/project/bin_jip/miniconda3/envs/biomarker/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Number of cells labeled as 'Doublet': 5690
Number of cells after removing 'Doublet': 253862


In [2]:
asthma_baseline.write_h5ad('./asthma/asthma_baseline_filtered_no_doublets.h5ad')


In [7]:
import scanpy as sc
adata_path = "asthma/asthma_baseline_data.h5ad"
adata2_path = "/data2/project/bin_jip/Biomarker/data/asthma/asthma_baseline_filtered_no_doublets.h5ad"
adata = sc.read_h5ad(adata_path)
adata2 = sc.read_h5ad(adata2_path)

In [8]:
print(adata.obs['CellType_minor'].value_counts())
print(adata2.obs['CellType_minor'].value_counts())

CellType_minor
CD14         45647
NK           45387
CD4 Naive    37681
CD8 TEM      22636
CD4 TEM      20390
B Naive      18122
CD4 TCM      15886
B Memory     12358
CD8 Naive    10790
CD8 TCM      10290
Treg          6262
Doublet       5690
CD16          4941
cDC           2726
pDC            575
ILC            171
Name: count, dtype: int64
CellType_minor
CD14         45647
NK           45387
CD4 Naive    37681
CD8 TEM      22636
CD4 TEM      20390
B Naive      18122
CD4 TCM      15886
B Memory     12358
CD8 Naive    10790
CD8 TCM      10290
Treg          6262
CD16          4941
cDC           2726
pDC            575
ILC            171
Name: count, dtype: int64
